[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C12/E12-2_Temperature_Broche.ipynb)

# Température de broche heure par heure : battre la persistance avec le passé récent

## Le contexte

Huit broches de machine-outil sont instrumentées. Pour chacune, un relevé toutes les cinq minutes donne la vitesse de consigne, la température du liquide de refroidissement, la température de l'air de l'atelier et la température de la broche, mesurée par la sonde du palier avant. Le programme de vitesse est connu à l'avance, par paliers.

La broche chauffe et refroidit lentement. On voudrait prévoir sa température une heure à l'avance, soit 12 relevés, pour décider quand lancer une pièce de précision : le défaut de forme dû à la dilatation thermique dépend de cette température. La prévision la plus simple, dite persistance, répond que la température dans une heure sera celle d'aujourd'hui. Le passé récent permet de mieux faire.

La grandeur à prédire est un nombre en °C : c'est un problème de régression sur une série temporelle.

Défi : prédire `temperature_broche_C` une heure à l'avance et obtenir un R² supérieur à celui du modèle de départ sur la base de test. Le modèle de départ est une autorégression : une régression ridge sur les quatre derniers relevés de température connus ; il est ajusté sur les relevés d'apprentissage antérieurs d'une heure au moins au premier relevé de test de chaque broche, et atteint un R² de 0,248, alors que la persistance donne -0,182 : à vous de faire mieux. La base de test est formée, pour chaque broche, du dernier quart de ses relevés : jamais un tirage au hasard, car le futur ne doit pas passer dans l'apprentissage.

## Les données

Fichier `MecaIA-Temperature_Broche.xlsx`, dans le dossier `donnees` du dépôt : 480 relevés, 6 colonnes. Une ligne est un relevé de cinq minutes sur une broche.

| Colonne | Signification | Unité |
|---|---|---|
| `machine` | identifiant de la machine, m1 à m8, une broche par machine | catégorie |
| `horodatage` | date et heure du relevé, toutes les 5 min, 60 relevés par machine | date et heure |
| `vitesse_rpm` | vitesse de consigne de la broche (quelques cases vides) | tr/min |
| `temperature_liquide_C` | liquide de refroidissement à l'entrée de la broche | °C |
| `temperature_ambiante_C` | air de l'atelier près de la machine (quelques cases vides) | °C |
| `temperature_broche_C` | cible : température de la broche, sonde du palier avant | °C |

Origine : jeu simulé, échauffement thermique du premier ordre d'une broche, avec un programme de vitesse par paliers. Il enseigne une méthode, et aucune conclusion de procédé n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides, le pas de temps et le nombre de relevés par machine : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Temperature_Broche.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Temperature_Broche.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "temperature_broche_C"
df["horodatage"] = pd.to_datetime(df["horodatage"])
print("dimensions :", df.shape)
vides = df.isna().sum()
print("cases vides par colonne :")
print(vides.to_string())
print("relevés par machine :", df["machine"].value_counts().sort_index().tolist())
pas = df.groupby("machine")["horodatage"].diff().dropna()
print("pas de temps :", {str(k): int(n) for k, n in pas.value_counts().items()})
print("du", df["horodatage"].min(), "au", df["horodatage"].max())

Lecture du résultat : 480 relevés, 8 machines de 60 relevés chacune, au pas régulier de cinq minutes. La vitesse de consigne a 20 cases vides et la température ambiante 22. Chaque machine est suivie une journée différente : les séries sont indépendantes, mais le temps reste l'ordre à respecter à l'intérieur de chacune.

### Étape 2 : ranger le temps et mettre le test de côté

On trie les relevés de chaque machine par date. Le dernier quart de chaque série, soit les 15 derniers relevés de chaque broche, est mis de côté avant tout apprentissage : il joue le rôle de l'avenir. L'apprentissage ne contient, pour chaque machine, que des relevés antérieurs à ceux du test. Aucun mélange des lignes n'est permis.

In [ ]:
df = df.sort_values(["machine", "horodatage"])
effectif = df.groupby("machine")["horodatage"].transform("size")
rang_depuis_la_fin = df.groupby("machine").cumcount(ascending=False)      # 0 : dernier relevé de la machine
est_test = rang_depuis_la_fin < np.ceil(0.25 * effectif)
print("apprentissage :", int((~est_test).sum()), "relevés ; test :", int(est_test.sum()), "relevés")
m1 = df[df["machine"] == "m1"]
print("machine m1, dernier relevé d'apprentissage :", m1.loc[~est_test, "horodatage"].max())
print("machine m1, premier relevé de test :", m1.loc[est_test, "horodatage"].min())

### Étape 3 : regarder une série

La température de la première machine, avec la vitesse de consigne. La zone grisée est la partie mise de côté.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 4.6), sharex=True)
axes[0].plot(m1["horodatage"], m1[cible], color="0.1")
axes[0].set_ylabel("température de broche (°C)")
axes[1].step(m1["horodatage"], m1["vitesse_rpm"].ffill(), where="post", color="0.4")
axes[1].set_ylabel("vitesse de consigne (tr/min)")
debut_test = m1.loc[est_test, "horodatage"].min()
for axe in axes:
    axe.axvspan(debut_test, m1["horodatage"].max(), color="0.85")
    axe.grid(alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

Lecture du résultat : la température suit les paliers de vitesse avec un retard. Après un changement de palier, elle monte ou descend progressivement vers une valeur d'équilibre, comme un système du premier ordre. Pour prévoir une heure à l'avance, le seul passé de la température ne prévient pas du prochain changement de palier : le programme de vitesse, lui, le fait.

### Étape 4 : construire les décalages

Pour prévoir la température au relevé t, on n'a le droit qu'aux relevés jusqu'à t moins une heure, soit 12 relevés plus tôt. On construit donc, pour chaque machine, la température décalée de 12, 13, 14 et 15 relevés. Les premiers relevés d'une série n'ont pas de passé : on y prolonge le premier relevé de la machine, la broche partant à froid. La vitesse de consigne, connue à l'avance, garde sa valeur précédente quand une case est vide ; en tête de série, elle reçoit la médiane du signal sur l’ajustement seul.

In [ ]:
HORIZON = 12                                                     # relevés de cinq minutes, soit une heure
par_machine = df.groupby("machine")


def decaler(colonne, pas):
    """Valeur de la colonne `pas` relevés plus tôt, la même machine ; premier relevé de la série en début de série."""
    return par_machine[colonne].shift(pas).fillna(par_machine[colonne].transform("first"))


apprentissage = df[~est_test]
n_app = apprentissage.groupby("machine")["horodatage"].transform("size")
rang_app = apprentissage.groupby("machine").cumcount(ascending=False)
valid = apprentissage.index[rang_app < np.ceil(0.25 * n_app)]
debut = df.loc[valid].groupby("machine")["horodatage"].min()
origine = apprentissage["machine"].map(debut) - pd.to_timedelta(5 * HORIZON, unit="min")
sous_train = apprentissage.index[apprentissage["horodatage"] <= origine]
vitesse_brute = df["vitesse_rpm"].copy()
depart = vitesse_brute.loc[sous_train].median()
df["vitesse_rpm"] = par_machine["vitesse_rpm"].ffill().fillna(depart)
X_lags = pd.DataFrame({f"T_moins_{pas}": decaler(cible, pas) for pas in range(HORIZON, HORIZON + 4)})
y = df[cible]
X_train, X_test = X_lags[~est_test], X_lags[est_test]
y_train, y_test = y[~est_test], y[est_test]
print(X_lags.head(3).round(1).to_string())

### Étape 5 : construire le modèle de départ

Le modèle de départ est une autorégression : une régression ridge qui prédit la température d'après les quatre dernières températures connues. Le ridge, qui freine un peu les coefficients, convient bien à des décalages voisins, très corrélés entre eux. Il apprend sur les relevés d'apprentissage antérieurs d'une heure au moins au premier relevé de test de leur broche, soit 272 relevés sur 360 : à l'origine de la prévision du premier relevé de test, les températures de l'heure précédente ne sont pas encore connues, et le modèle ne doit pas s'en servir. C'est le même écart d'un horizon que celui de la validation chronologique du « pour aller plus loin ».

In [ ]:
# Votre code :

### Étape 6 : mesurer sur la base de test

Le modèle prédit des relevés qu'il n'a jamais vus. On lit le coefficient de détermination R² et deux erreurs en °C, la racine de l'erreur quadratique moyenne (RMSE) et l'erreur absolue moyenne (MAE). Deux témoins sans apprentissage servent de comparaison : la persistance, qui répond la dernière température connue, et la moyenne de l'apprentissage.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

y_pred = model.predict(X_test)
score_depart = r2_score(y_test, y_pred)
score_persistance = r2_score(y_test, X_test[f"T_moins_{HORIZON}"])
score_moyenne = r2_score(y_test, np.full(len(y_test), y_train.mean()))
print(f"R² : {score_depart:.3f}")
print(f"RMSE : {root_mean_squared_error(y_test, y_pred):.2f} °C")
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f} °C")
print(f"R² de la persistance : {score_persistance:.3f}")
print(f"R² de la moyenne de l'apprentissage : {score_moyenne:.3f}")

Lecture du résultat : le R² est de 0,248 et l'erreur absolue moyenne de 7,34 °C. La persistance fait nettement moins bien (-0,182), moins bien même que la moyenne (-0,107) : à une heure, la température a eu le temps de changer de palier. L'autorégression améliore la persistance, mais reste loin d'une bonne prévision, puisque le passé de la température ignore le programme de vitesse.

## Pour aller plus loin : donner au modèle le programme de vitesse

Le modèle de départ de l'étape 5 reste le score de référence du défi. Les deux cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. La première ajoute des variables que l'on connaît à l'instant de la prévision : la vitesse de consigne à l'instant visé, sa moyenne sur la dernière heure et sa puissance 1,5 (l'échauffement croît à peu près ainsi avec la vitesse), puis la température du liquide et de l'air une heure plus tôt. Elle compare les ensembles de variables et un gradient boosting sur une validation chronologique interne à l'apprentissage : le dernier quart de l'apprentissage de chaque machine sert de validation, avec un écart d’un horizon entre les cibles d’ajustement et de validation. La seconde réajuste le modèle retenu avec le même écart d'un horizon que le modèle de départ, puis le mesure une seule fois sur le test. Les lectures sont imprimées par les cellules elles-mêmes.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

## Le score de référence

Le score publié est le R² du modèle de départ de l'étape 5 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = r2_score(y_test, model.predict(X_test))
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE r2 = %.3f" % score)

## Synthèse

1. Une série temporelle se découpe dans l'ordre du temps : le test est le dernier quart de chaque série, et l'apprentissage ne contient que des relevés antérieurs. Un tirage au hasard ferait passer le futur dans l'apprentissage. Le réajustement final garde, comme la validation, un écart d'une heure avec le premier relevé de test.
2. À une heure, la persistance ne vaut rien (-0,182) : la température a changé de palier entre-temps. L'autorégression sur les quatre derniers relevés donne 0,248.
3. Ce que le passé de la température ne dit pas, le programme de vitesse le dit : en validation chronologique, l'ajout de la vitesse de consigne fait passer le R² de -0,440 à 0,145.
4. Des variables construites d'après la physique, la vitesse moyenne de la dernière heure et sa puissance 1,5, font monter la validation à 0,514 avec une régression linéaire. Le gradient boosting, qui trouve seul les non-linéarités, fait mieux encore (0,803).
5. Le modèle retenu par la validation chronologique est « gradient boosting, toutes les variables » ; mesuré une seule fois sur le test, il donne un R² de 0,889. La base de test compte 120 relevés de huit séries : le chiffre reste fragile.